In [5]:
import copy
import json
from pathlib import Path

import numpy as np
import hemo1d as hd

# Experiment 3: CoW with AComm hypoplasia

In [ ]:
METHOD = "CG"
DG_TIME_SCHEME = "rk2"

DEGREE = 1
H = 0.0625
DT = 1.0e-5
T_END = 2.0
RECORD_EVERY = 100

HYPOPLASTICITY = True
HYPOPLASTICITY_FACTOR = 0.05

HEART_RATE = 1.2
RAMP_TIME = 0.2

CONFIG_PATH = Path("./configs/CoW_normal.json")
HYPOPLASIA_CONFIG_PATH = Path("./configs/CoW_ACA_hypoplastic.json")

### Load CoW configuration

In [16]:
cow_config = json.loads(CONFIG_PATH.read_text())

area_ACA = cow_config["vessels"]["ACA"]["initial_area"]

hypoplasia_config = copy.deepcopy(cow_config)
hypoplasia_config["vessels"]["ACA"]["initial_area"] = area_ACA * HYPOPLASTICITY_FACTOR

# Save the hypoplastic configuration to a new file
HYPOPLASIA_CONFIG_PATH.write_text(json.dumps(hypoplasia_config, indent=4))

9404

### Set the pressure inlets

In [14]:
inflow_data = {
    "BAS" : {
        "mean_mmhg": 84.0,
        "amp1_mmhg": 11.0,
        "amp2_mmhg": 3.0,
        "heart_rate": HEART_RATE,
        "ramp_time": RAMP_TIME,
    },
    "L-ICA_I": {
        "mean_mmhg": 85.0,
        "amp1_mmhg": 14.0,
        "amp2_mmhg": 4.0,
        "heart_rate": HEART_RATE,
        "ramp_time": RAMP_TIME,
    },
    "R-ICA_I": {
        "mean_mmhg": 85.0,
        "amp1_mmhg": 14.0,
        "amp2_mmhg": 4.0,
        "heart_rate": HEART_RATE,
        "ramp_time": RAMP_TIME,
    }
}

### Set up models

In [ ]:
model_CoW_normal = hd.load_from_config(CONFIG_PATH)
model_CoW_hypoplastic = hd.load_from_config(HYPOPLASIA_CONFIG_PATH)

# Inlets
for vessel_name, inflow_params in inflow_data.items():
    p_in = hd.create_arterial_pressure_inflow(**inflow_params)
    model_CoW_normal.set_inlet(vessel_id=vessel_name, kind="pressure", function=p_in)
    model_CoW_hypoplastic.set_inlet(vessel_id=vessel_name, kind="pressure", function=p_in)

# Solver settings
for model in [model_CoW_normal, model_CoW_hypoplastic]:
    model.set_solver(
        method=METHOD,
        poly_order=DEGREE,
        h=H,
        dt=DT,
        dg_time_scheme=DG_TIME_SCHEME,
        record_every=RECORD_EVERY,
    )

# Set probes